# Summary of: AsymmeTree – A Flexible Python Package for the Simulation of Complex Gene Family Histories

## Overview

AsymmeTree is a Python library for simulating gene family histories (GFHs) under realistic evolutionary scenarios. It integrates the simulation of dated species trees, gene trees with duplications, losses, horizontal gene transfer (HGT), gene conversion, and finally nucleotide or amino acid sequences. The package is designed to be used as part of complex benchmarking pipelines, providing direct programmatic access to simulated data rather than only static output files.

## Key Features

- **Species tree simulation** – Yule model, constant‑rate birth‑death process (BDP), episodic BDP (EBDP), and the **innovation model** (unique to AsymmeTree).

- **Gene tree simulation** – Simultaneous handling of duplications, losses, additive/replacing HGT, gene conversion, and optional multifurcations. Rates are user‑defined; extinction of essential genes can be prevented.

- **Evolution rate heterogeneity** – Three levels: gene‑family‑specific baseline rates, species‑specific relaxed clock (lognormal model), and **paralog‑specific asymmetry** (neofunctionalisation / subfunctionalisation). Asymmetry is modelled via a Gamma distribution fitted to yeast whole‑genome duplication data.

- **Sequence simulation** – Continuous‑time Markov models for substitutions (nucleotide: JC69, K80, GTR; amino acid: Dayhoff, BLOSUM62, JTT, WAG, LG, or custom PAML‑format models). Indels follow a Zipfian distribution; among‑site rate heterogeneity ($+\Gamma$ model) and invariant sites ($+\bar{I}$) are supported.

- **True alignment** – The package can reconstruct the true multiple sequence alignment (MSA) even with indels, by tracking site identifiers.

## Simulation Pipeline (5 steps)

1. Generate a dated species tree $S$ (with optional loss branches).

2. Simulate a dated gene tree $T$ along $S$ using a birth‑death process with events: duplication, loss, HGT (additive/replacing), gene conversion.

3. Introduce rate heterogeneity:
   - Baseline family rate (e.g., uniform, Gamma, exponential).
   - Species‑specific multipliers via a lognormal relaxed clock:  
     $\log(r_v) \sim \mathcal{N}\bigl(\text{mean adjusted so that } \mathbb{E}[r_v]=r_u,\; \beta t\bigr)$, where $t$ is divergence time.
   - Paralog‑specific multipliers: after duplication, one or both copies become “divergent” with rate multiplier $1+x$, where $x$ is drawn from a Gamma distribution (shape $k=0.5$, scale $\theta=2.2$) fitted to asymmetry parameter $R'$:  
     $R' = \frac{\max(K_a,K_b)}{\min(K_a,K_b)} = 1 + X$.

4. Prune branches that lead only to losses.

5. Evolve sequences along the pruned gene tree using substitution, indel, and heterogeneity models.

## Validation and Performance

- Species tree ages match those from **TreeSim** (Mann‑Whitney $U$ test, $p>0.05$).

- HGT distance bias (inverse, exponential) produces expected lower divergence times.

- Lognormal rates are unbiased and leaf log‑rates are normally distributed.

- Simulated $R'$ values reproduce the input Gamma distribution.

- Sequence distances re‑estimated from simulated data (JC69, K80, WAG, JTT) correlate well with true distances.

- AsymmeTree is slower than C/C++ tools (Seq‑Gen, INDELible) but faster than **Pyvolve** (Python).

In [16]:
# Import packages
import asymmetree.treeevolve as te   # Tree simulation
import asymmetree.seqevolve as se    # Sequence evolution
import asymmetree.analysis as ana    # Analysis (orthology, etc.)

In [17]:
## To convert trees to Newick format, useful for external visualisation:
from asymmetree.utils.phylogenetic_trees import to_newick

import asymmetree.treeevolve as te
from asymmetree.utils.phylogenetic_trees import to_newick

In [18]:
# Yule model (speciation only, no extinction)
tree_yule = te.species_tree_n(10, model="yule", birth_rate=1.0)
print("Yule tree (Newick):")
print(to_newick(tree_yule))

Yule tree (Newick):
((((((18:0.06995042378104332,19:0.06995042378104332)17:0.1137047359181862,16:0.18365515969922952)6:0.7870055482937002,7:0.9706607079929297)4:0.4562424200476509,((12:0.3970104370095682,13:0.3970104370095682)10:0.02652231138386396,11:0.42353274839343213)5:1.0033703796471485)3:0.11376881729066479,((14:0.27461688331591283,15:0.27461688331591283)9:0.256432196618827,8:0.5310490799347398)2:1.0096228653965056)1:0.4625542698083942)0:0.0;


In [19]:
# Episodic birth‑death model (EBDP)
# List of tuples: (time, birth_rate, death_rate, sampling_frac)
episodes = [(1.0, 0.3, 0.8, 0.0), (0.9, 0.4, 0.6, 0.3)]
tree_ebdp = te.species_tree_n(10, model="EBDP", episodes=episodes)
print("\nEBDP tree (Newick):")
print(to_newick(tree_ebdp))


EBDP tree (Newick):
((((((((((0:0.3272528050189843,17:0.027252805018984305)27:1.011085058629514,17:1.0383378636484981)53:0.3020387508553235,(((10:0.19199687821464995,12:0.1791314226101731)14:0.5121415403600439,17:0.40413841857469385)38:0.3022403895529612,17:0.706378808127655)49:0.6339978063761667)56:0.3848576702093227,((((((5:0.7812494644395825,((9:0.32862075099469606,17:0.028620750994696076)28:0.0741894198490704,17:0.10281017084376648)30:0.3784392935958161)43:0.05416851186150351,42:0.0592042950131908)44:0.04892493888486105,17:0.5843429151859472)45:0.12080352763413227,16:0.740458002701875)48:0.23215882793039833,8:1.2373052707504777)51:0.7392014974150922,29:1.6334875871511891)61:0.048727516547574456)62:0.2851731402281885,(((10:1.5474082933133964,(13:0.5717716518795396,32:0.08175502199396723)35:0.9566324202983211)55:0.16867798783944865,54:0.20565432135246775)58:0.37280795909681896,(17:0.6312669158455857,36:0.3374627055097459)47:1.1576273244040782)64:0.22151318469166892)67:1.309707892698

In [20]:
# We can also simulate conditioning on the total age of the tree with species_tree_age:
tree_age = te.species_tree_age(2.0, model="yule", birth_rate=1.0)
print(to_newick(tree_age))

#### Simulating a gene tree with complex events ####

##  It allows a gene family to evolve along a species tree, including duplications, losses, horizontal gene transfers (HGT) and conversions.

# 1. Species tree with 10 leaves and 1 time unit
S = te.species_tree_n_age(10, 1.0)

# 2. Simulate gene tree with duplications, losses and HGT
TGT = te.dated_gene_tree(
    S,
    dupl_rate=1.0,      # duplication rate
    loss_rate=1.0,      # loss rate
    hgt_rate=0.2,       # horizontal transfer rate
    gc_rate=0.2,        # gene conversion rate
    prohibit_extinction="per_species"
)
print(to_newick(TGT))

(((4:1.0844534246976556,5:1.0844534246976556)2:0.6055550248176886,((8:0.004668655103702513,9:0.004668655103702513)7:0.6013623268667287,6:0.6060309819704313)3:1.083977467544913)1:0.30999155048465576)0:0.0;
(((((((31<12-15>:0.1123651596826973,30<14>:0.12125045374484682)24:0.06935327090100772,(46<18>:0.03822683179427184,47<19>:0.03822683179427184)25:0.1523768928515827)10:0.08637338380005385,(((32<14>:0.12125045374484682,33<15>:0.12125045374484682)28:0.022006146052761924,(34<12-14>:0.01636176487361536,35<15>:0.12125045374484682)29:0.022006146052761924)26:0.0473471248482458,((50<19>:0.001912814173013368,51<19>:0.001912814173013368)49:0.03631401762125847,48<18>:0.03822683179427184)27:0.1523768928515827)11:0.08637338380005385)7:0.05603173922480881,6<2-4>:0.05603173922480881)5:0.018434256441311803,((19<5-10>:0.08588685626356968,(38<16>:0.07566152464023006,39<17>:0.07566152464023006)18:0.12985994806190332)8:0.11879160802536193,((40<16>:0.07566152464023006,41<17>:0.07566152464023006)20:0.1298599

In [21]:
#### Add heterogeneity in evolutionary rates ####

# After generating the gene tree, we can apply rate heterogeneity to its branches using te.rate_heterogeneity():

TGT_het = te.rate_heterogeneity(
    TGT,
    S,
    base_rate=1.0,
    autocorr_variance=0.2,
    rate_increase=("gamma", 0.5, 2.2),  # gamma distribution for increments
    CSN_weights=(1, 1, 1)
)
print(to_newick(TGT_het))

(((((((31<12-15>:0.06770615196066056,30<14>:0.07611196753064021)24:0.04893617022021648,(46<18>:0.02697699548881386,47<19>:0.027138538466506382)25:0.11285441997975619)10:0.07029816871409401,(((32<14>:0.4526892640837011,33<15>:0.07689518988093211)28:0.01552770641593811,(34<12-14>:0.04543605125426234,35<15>:0.08985309311442176)29:0.06559879318214537)26:0.033408474191776456,((50<19>:0.0024450917993640107,51<19>:0.0013579723606961477)49:0.025780566105810237,48<18>:0.02697699548881386)27:0.11285441997975619)11:0.07029816871409401)7:0.045603500570129026,6<2-4>:0.045603500570129026)5:0.015003400496964244,((19<5-10>:0.0755331893481154,(38<16>:0.06495555984364117,39<17>:0.069862532022711)18:0.11119044390274659)8:0.102126796013096,((40<16>:0.07014574509347558,41<17>:0.23450348317460545)20:0.1457132284101069,21<10>:0.2484274941800023)9:0.8569759911926911)4:0.023324058071772655)2:0.1368758891395216,(((16<8>:0.14981108709359023,17<9>:0.17034865724061374)14:0.0030372847795697754,(((44<10>:0.050689972

The parameters autocorr_variance and rate_increase control how rates vary along the tree.

#### Simulating DNA or protein sequences ####

Once we have a tree, with branch lengths, we can evolve sequences along it using the seqevolve submodule.

In [22]:
import asymmetree.seqevolve as se

# Substitution model (amino acids with WAG matrix)
subst_model = se.SubstModel("a", "WAG")

# Indel model (insertions and deletions)
indel_model = se.IndelModel(0.01, 0.01, length_distr=("zipf", 1.821))
# You can also use a negative binomial distribution:
# indel_model = se.IndelModel(0.01, 0.01, length_distr=('negative_binomial', 1, 0.5))

## We create the “evolver” and simulate it along the tree.

# Initialise evolver
evolver = se.Evolver(subst_model, indel_model=indel_model, gillespie=False)

In [23]:
# Example tree
T = te.species_tree_n_age(5, 1.0)

# Evolve sequences starting from an initial length of 150 bp
evolver.evolve_along_tree(T, start_length=150)

## Save and visualise results

# Save to FASTA (including internal nodes)
evolver.write_sequences("sequences.fasta", include_inner=True)

# Obtain true alignment (without gaps)
alignment = evolver.true_alignment(write_to="alignment.aln")

# Show some sequences
for node, seq in list(evolver.sequences.items())[:5]:
    print(node.label, subst_model.to_sequence(seq))

# The package uses matplotlib to visualise trees.


0 MDIANYHAGKKPARMKKVLKAKEDSFQTKSNVSAFGLNDMNESDDTTLLALFSVLVATSGVSCVSTMYRIRFWGPSATTVGPVEATYAIHYDYGDRCESPKIWDDNRGAPAGQEGAGGRAVPDTEALQPLGSRFGDGPEKGLGFRHVTEQ
1 MDIANYSAGKKPARMKKVLKAKDDSFETKSNVSAFGLTDMNESDHTTLLALFSVLVATSMVSCVSTMYRIRLWDPSATAVGPVEATYAIHYDYDARVENPKIWDDNRGAPAGQEGAGVRKTPDSEALQPLGSRYGDGPEKGLGFRHVTEQ
3 MDIANYSAGKKPARMKKVLKAKDDSFETKSNVSAFGLTDMNESDHTTLLALFSVLVATSMVSCVSTMYRIRLWDPSATAVGPVEATYAIHYGYDARVENPKIWDDNRGAPAGQEGAGVQKTPDSEVLQPLGSRYGDGPEKGLGFRHVTEQ
4 MHLSNYGTGRQFGRKTKVLNAKDDNFETKTSLSAFGLMDTNEGDHTTHLALFTVLVATAFVSCVNTMWRIRLWDPSDTAPKRIDATYARHSGYDESIENPRIYEGRGAPAGHEGAGVWKTPDRAVFSPLGSKYGDGPAKSLGFKEVTEN
8 AFQSNYGQGGIYGQRTKVQNTLPDNFETKTNLGAFGLVLDSEEGDHTTHLARFNILNEILNCVNTLKRIRVWDKSDAATKHLEAEYARHPGWDKGLEAPRFYDGRGKPAGHEGAKVWKTPDRAVLAPLG
